# Projection V2 — Colab CPU preparation

Accelerator **None**. Đặt ba zip `projection-v2-cpu-inputs.zip`, `projection-v2-heads.zip`, `grace-h1-joern-runtime.zip` vào MyDrive/GRACE-projection-v2-transfer. Không cần Vertex credential hoặc GPU. I/O parse chạy ở `/content`, Drive chỉ dùng cho zip đầu vào/checkpoint/output. Không chạy hai notebook cùng lúc vào cùng thư mục transfer.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os,sys,subprocess,shutil
ROOT=Path('/content/grace-projection-v2')
TRANSFER=Path('/content/drive/MyDrive/GRACE-projection-v2-transfer')
INPUT_ZIP=TRANSFER/'projection-v2-cpu-inputs.zip'
HEADS_ZIP=TRANSFER/'projection-v2-heads.zip'
JOERN_ZIP=TRANSFER/'grace-h1-joern-runtime.zip'
CHECKPOINT=TRANSFER/'projection-v2-cpu-checkpoint.zip'
for p in (INPUT_ZIP,HEADS_ZIP,JOERN_ZIP):
    if not p.is_file(): raise FileNotFoundError(p)


In [ ]:
import hashlib, json, sys, zipfile
from pathlib import Path, PurePosixPath
def bootstrap_sources(root, archive_path, expected_kind):
    root.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        manifest=json.loads(archive.read('bundle_manifest.json'))
        if manifest['schema']!='grace_projection_v2_bundle_v1' or manifest['kind']!=expected_kind:
            raise ValueError('Wrong V2 input bundle')
        for entry in manifest['entries']:
            name=entry['path'];p=PurePosixPath(name)
            source_file=name.startswith('grace_h1/') and p.suffix=='.py'
            dependency_file=name in ('requirements.txt','requirements-cpu.txt','requirements-training.txt')
            if not (source_file or dependency_file): continue
            if p.is_absolute() or '..' in p.parts or ':' in name or '\\' in name:
                raise ValueError('Unsafe source path')
            data=archive.read(name)
            if len(data)!=entry['bytes'] or hashlib.sha256(data).hexdigest()!=entry['sha256']:
                raise ValueError('Source checksum mismatch')
            target=root/name;target.parent.mkdir(parents=True,exist_ok=True)
            if target.exists() and target.read_bytes()!=data:
                raise ValueError('Source changed: use a fresh cloud ROOT')
            target.write_bytes(data)
    if str(root) not in sys.path: sys.path.insert(0,str(root))

bootstrap_sources(ROOT,INPUT_ZIP,'cpu_inputs')
subprocess.check_call([sys.executable,'-m','pip','install','-r',str(ROOT/'requirements-cpu.txt'),'tqdm>=4.66'])
from grace_h1.portable_v2 import import_bundle,create_bundle
from grace_h1.portable import import_bundle as import_joern_bundle
print(import_bundle(ROOT,INPUT_ZIP,'cpu_inputs'))
print(import_bundle(ROOT,HEADS_ZIP,'heads'))
print(import_joern_bundle(ROOT,JOERN_ZIP,'joern_runtime'))
subprocess.check_call(['apt-get','update','-qq'])
subprocess.check_call(['apt-get','install','-y','-qq','openjdk-21-jre-headless'])
os.environ['JAVA_HOME']='/usr/lib/jvm/java-21-openjdk-amd64'
os.environ['PATH']=os.environ['JAVA_HOME']+'/bin:'+os.environ['PATH']
os.environ['JOERN_HOME']=str(ROOT/'tools/joern/joern-cli')
if CHECKPOINT.exists(): print(import_bundle(ROOT,CHECKPOINT,'checkpoint'))
from grace_h1.experiment_v2 import load_config,prepare_all
config=load_config(ROOT)
print('workers:',config['pipeline']['graph_workers'],'datasets/splits:',config['datasets'],config['splits'])


## Chuẩn bị và theo dõi graph

Thanh tiến độ cập nhật theo graph hoàn tất, có cache hit, parse mới và ID đang chạy. Giữ 3 workers như config hiện tại. Checkpoint theo mốc 500 graph có thay đổi, cuối arm và khi interrupt; không checkpoint sau từng mẫu. Nếu cần đổi workers/config/source, tạo lại CPU-inputs tương ứng trước khi resume.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from threading import Lock
from tqdm.auto import tqdm
from grace_h1 import pipeline as pipeline_module
class ProgressPool(ThreadPoolExecutor):
    def map(self,fn,rows,timeout=None,chunksize=1):
        lock=Lock();state={'cache':0,'parsed':0};active={}
        bar=tqdm(total=len(rows),desc='Graph preparation',unit='graph',dynamic_ncols=True)
        def tracked(row):
            h=row['code_sha256'];folder=ROOT/'cache/graphs'/h
            cached=(folder/'features.json.gz').exists() or (folder/'features.json').exists()
            with lock:
                active[h]=row['record_id']
                bar.set_postfix_str(f"cache {state['cache']} | parsed {state['parsed']} | đang chạy {len(active)}: {str(list(active.values())[:2])[:100]}")
            try:
                result=fn(row)
                with lock:
                    state['cache' if cached else 'parsed']+=1
                    bar.update(1)
                return result
            finally:
                with lock:
                    active.pop(h,None)
                    bar.set_postfix_str(f"cache {state['cache']} | parsed {state['parsed']} | đang chạy {len(active)}: {str(list(active.values())[:2])[:100]}")
        def results():
            try: yield from super(ProgressPool,self).map(tracked,rows,timeout=timeout,chunksize=chunksize)
            finally: bar.close()
        return results()
pipeline_module.ThreadPoolExecutor=ProgressPool
def checkpoint():
    local=ROOT/'bundles/projection-v2-cpu-checkpoint.zip'
    info=create_bundle(ROOT,local,'checkpoint')
    target=CHECKPOINT.with_name(CHECKPOINT.name+'.tmp')
    shutil.copyfile(local,target);os.replace(target,CHECKPOINT)
    tqdm.write(f"Checkpoint saved: {CHECKPOINT} ({info['bytes']} bytes)")
try:
    rows=prepare_all(ROOT,config,checkpoint_callback=checkpoint)
finally:
    checkpoint()
print(rows)


In [ ]:
output=ROOT/'bundles/projection-v2-cpu-prepared.zip'
print(create_bundle(ROOT,output,'prepared'))
shutil.copyfile(output,TRANSFER/output.name)
from google.colab import files
files.download(str(output))
print('Đặt projection-v2-cpu-prepared.zip vào bundles/ local, mở lại full-pipeline.ipynb và chạy từ đầu.')
